In [ ]:
import os
from pathlib import Path

# Kaggle paths remain the defaults when running in Kaggle; local users can set
# PEST_DATASET_ROOT and PEST_WORK_DIR before starting Jupyter.
default_dataset_root = (
    Path("/kaggle/input/insect-village-synthetic-dataset")
    if Path("/kaggle/input").is_dir()
    else Path("data/insect-village-synthetic-dataset")
)
default_work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("artifacts")
DATASET_ROOT = Path(os.environ.get("PEST_DATASET_ROOT", str(default_dataset_root))).expanduser()
WORK_DIR = Path(os.environ.get("PEST_WORK_DIR", str(default_work_dir))).expanduser()
ANNOTATION_DIR = DATASET_ROOT / "ImageClassesCombinedWithCOCOAnnotations"
IMAGE_DIR = ANNOTATION_DIR / "images_raw"
COCO_JSON = ANNOTATION_DIR / "coco_instances.json"
OUTPUT_DIR = WORK_DIR / "yolo_dataset"

if not IMAGE_DIR.is_dir() or not COCO_JSON.is_file():
    raise FileNotFoundError(
        "Expected the Insect Village dataset at "
        f"{DATASET_ROOT}. Set PEST_DATASET_ROOT to the extracted dataset directory."
    )
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset root: {DATASET_ROOT}")
print(f"Generated files will be written under: {WORK_DIR}")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image_files = sorted(
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".png", ".jpeg"}
)[:9]

plt.figure(figsize=(12, 12))
for idx, image_path in enumerate(image_files):
    with Image.open(image_path) as image:
        plt.subplot(3, 3, idx + 1)
        plt.imshow(image.copy())
        plt.title(image_path.name, fontsize=8)
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

labels_dir = OUTPUT_DIR / "labels"
yolo_images_dir = OUTPUT_DIR / "images"
labels_dir.mkdir(parents=True, exist_ok=True)
yolo_images_dir.mkdir(parents=True, exist_ok=True)

with COCO_JSON.open("r", encoding="utf-8") as f:
    coco_data = json.load(f)

categories = coco_data.get("categories", [])
if not categories:
    raise ValueError("The COCO annotation file does not define any categories.")
category_id_map = {category["id"]: index for index, category in enumerate(categories)}
category_names = [category["name"] for category in categories]
print(f"Total classes: {len(category_names)}")
print(f"Class names: {category_names}")

annotations_by_image = {}
for annotation in coco_data.get("annotations", []):
    annotations_by_image.setdefault(annotation["image_id"], []).append(annotation)

missing_images = []
for image_info in tqdm(coco_data.get("images", []), desc="Converting COCO annotations"):
    relative_name = Path(image_info["file_name"])
    if relative_name.is_absolute() or ".." in relative_name.parts:
        raise ValueError(f"Unsafe image path in annotation file: {image_info['file_name']!r}")
    source_image = IMAGE_DIR / relative_name
    if not source_image.is_file():
        missing_images.append(image_info["file_name"])
        continue

    destination_image = yolo_images_dir / relative_name
    destination_image.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_image, destination_image)

    width, height = image_info["width"], image_info["height"]
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid image dimensions for {image_info['file_name']!r}.")
    label_lines = []
    for annotation in annotations_by_image.get(image_info["id"], []):
        category_id = annotation["category_id"]
        if category_id not in category_id_map:
            raise ValueError(f"Unknown category id {category_id!r} in COCO annotations.")
        x, y, box_width, box_height = annotation["bbox"]
        x_center = (x + box_width / 2) / width
        y_center = (y + box_height / 2) / height
        label_lines.append(
            f"{category_id_map[category_id]} {x_center:.6f} {y_center:.6f} "
            f"{box_width / width:.6f} {box_height / height:.6f}"
        )

    label_path = labels_dir / relative_name.with_suffix(".txt")
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(label_lines), encoding="utf-8")

print(f"Conversion complete. Missing images: {len(missing_images)}")
if missing_images:
    print("First missing image names:", missing_images[:10])


In [ ]:
import random
import shutil
from pathlib import Path

images_dir = OUTPUT_DIR / "images"
labels_dir = OUTPUT_DIR / "labels"
train_images_dir = images_dir / "train"
val_images_dir = images_dir / "val"
train_labels_dir = labels_dir / "train"
val_labels_dir = labels_dir / "val"

image_files = sorted(
    path for path in images_dir.rglob("*")
    if (
        path.is_file()
        and path.relative_to(images_dir).parts[0] not in {"train", "val"}
        and path.suffix.lower() in {".jpg", ".png", ".jpeg"}
    )
)
if len(image_files) < 2:
    raise ValueError("At least two images are required to create train and validation splits.")

# A fixed seed makes this split repeatable for a given file set.
random.Random(42).shuffle(image_files)
split_index = min(len(image_files) - 1, max(1, int(0.8 * len(image_files))))
train_files, val_files = image_files[:split_index], image_files[split_index:]

# These four directories are generated exclusively by this notebook; recreate them
# to prevent stale files from appearing in a subsequent training run.
for generated_dir in (train_images_dir, val_images_dir, train_labels_dir, val_labels_dir):
    if generated_dir.exists():
        shutil.rmtree(generated_dir)
    generated_dir.mkdir(parents=True, exist_ok=True)

for split_files, split_images_dir, split_labels_dir in (
    (train_files, train_images_dir, train_labels_dir),
    (val_files, val_images_dir, val_labels_dir),
):
    for image_path in split_files:
        relative_name = image_path.relative_to(images_dir)
        label_relative_name = relative_name.with_suffix(".txt")
        source_label = labels_dir / label_relative_name
        if not source_label.is_file():
            raise FileNotFoundError(f"Missing YOLO label file for {image_path.name}: {source_label}")
        destination_image = split_images_dir / relative_name
        destination_label = split_labels_dir / label_relative_name
        destination_image.parent.mkdir(parents=True, exist_ok=True)
        destination_label.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image_path, destination_image)
        shutil.copy2(source_label, destination_label)

print(f"Data split complete: {len(train_files)} training and {len(val_files)} validation images (seed=42).")


In [ ]:
import yaml

data_config = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": len(category_names),
    "names": category_names,
}

yaml_file = OUTPUT_DIR / "dataset.yaml"
with yaml_file.open("w", encoding="utf-8") as file:
    yaml.safe_dump(data_config, file, sort_keys=False)

print(f"Dataset configuration written to: {yaml_file}")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(
    data=str(yaml_file),
    epochs=170,
    imgsz=640,
    batch=16,
    name="yolov8_ONE_custom",
    project=str(WORK_DIR / "runs" / "detect"),
    seed=42,
)
model.save(str(WORK_DIR / "yolov8_trained_model.pt"))


In [ ]:
val_results = model.val()
metrics = val_results.results_dict
print(f"Precision: {metrics['metrics/precision(B)']:.4f}")
print(f"Recall: {metrics['metrics/recall(B)']:.4f}")
print(f"mAP@0.5: {metrics['metrics/mAP50(B)']:.4f}")
print(f"mAP@0.5:0.95: {metrics['metrics/mAP50-95(B)']:.4f}")


In [ ]:
import matplotlib.pyplot as plt

precision = metrics['metrics/precision(B)']
recall = metrics['metrics/recall(B)']
map_50 = metrics['metrics/mAP50(B)']
map_50_95 = metrics['metrics/mAP50-95(B)']

metrics_values = [precision, recall, map_50, map_50_95]
metrics_labels = ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']

plt.figure(figsize=(8, 6))
plt.bar(metrics_labels, metrics_values, color=['skyblue', 'lightgreen', 'salmon', 'yellow'])

plt.title('Model Evaluation Metrics')
plt.ylabel('Score')

for i, value in enumerate(metrics_values):
    plt.text(i, value + 0.01, f'{value:.4f}', ha='center')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cv2

metrics_img = WORK_DIR / "runs" / "detect" / "yolov8_ONE_custom" / "results.png"
if metrics_img.is_file():
    img = cv2.imread(str(metrics_img))
    if img is None:
        raise ValueError(f"Unable to read training plot: {metrics_img}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("YOLOv8 Training Metrics")
    plt.show()
else:
    print(f"Metrics image not found at: {metrics_img}. Run the training cell first.")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics_file = WORK_DIR / "runs" / "detect" / "yolov8_ONE_custom" / "results.csv"
metrics = pd.read_csv(metrics_file)

metrics.columns = metrics.columns.str.strip()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['train/box_loss'], label='Train Box Loss')
plt.plot(metrics['epoch'], metrics['val/box_loss'], label='Val Box Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Box Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['train/cls_loss'], label='Train Class Loss')
plt.plot(metrics['epoch'], metrics['val/cls_loss'], label='Val Class Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Class Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['metrics/mAP50(B)'], label='mAP@0.5')
plt.xlabel('Epoch')
plt.ylabel('mAP@0.5')
plt.title('mAP@0.5 Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
plt.xlabel('Epoch')
plt.ylabel('mAP@0.5:0.95')
plt.title('mAP@0.5:0.95 Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['metrics/precision(B)'], label='Precision')
plt.xlabel('Epoch')
plt.ylabel('Precision')
plt.title('Precision Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.ylabel('Recall')
plt.title('Recall Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(metrics['epoch'], metrics['train/dfl_loss'], label='Train Object Loss', color='brown')
plt.plot(metrics['epoch'], metrics['val/dfl_loss'], label='Val Object Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Object Loss')
plt.title('Object Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()